# **Parking Analysis in Seattle**

#### Seattle ranks among the top 10 most congested cities in the United States. The objective of our analysis is to help the Seattle Department of Transportation (Seattle SDOT) determine how to optimise parking spaces in the city. To do so, we will be adopting a three-step approach answering the following business questions:
1. How does utilisation vary across Seattle?
2. How does utilisation vary across the years?
3. Are prices aligned with the different demand study areas?

#### We will be using these 2 datasets to analyse the parking spaces in Seattle:

- Dataset 1: Annual Parking Study Data (csv file) - Covers the period of 2014 to 2019 (https://data.seattle.gov/Transportation/Annual-Parking-Study-Data/7jzm-ucez)
- Dataset 2: Paid Parking Transaction Data (json file) - Covers 1 week of data from 29 November 2025 to 5 December 2025 (https://data.seattle.gov/Transportation/Paid-Parking-Transaction-Data/gg89-k5p6/about_data)

## **1. Initiating Spark Session:** Embarking on our Analysis Process

In [ ]:
from pyspark.sql.session import SparkSession

spark_session = \
  SparkSession.builder\
              .appName("parking")\
              .getOrCreate()

print(f"This cluster relies on Spark '{spark_session.version}'")

In [ ]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
import os
# Tell Spark Session to access MinIO
spark_session.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.access.key", os.environ["MINIO_ACCESS_KEY"])
spark_session.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.secret.key", os.environ["MINIO_SECRET_KEY"])
spark_session.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
spark_session.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark_session.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://localhost:9000")

In [ ]:
csv_df = spark_session.read\
                         .option("header","true")\
                         .option("inferSchema","true")\
                         .csv("s3a://parking/csv/")
csv_df.limit(5).toPandas()

In [ ]:
json_df = spark_session.read\
                        .option("multiLine","true")\
                        .json("s3a://parking/json/")

json_df.limit(5).toPandas()

## **2. Data Cleaning:**  Everything Everywhere All at Once about Data Quality and Integrity

After both datasets are loaded from MinIO, we will now proceed to clean the datasets, before proceeding with our Exploratory Data Analysis (EDA).

In [ ]:
# Check datatype for each column
csv_df.printSchema()
json_df.printSchema()

From the above printSchema and the dataframe preview generated in Part 1 above, we can see that both datasets have 2 similar columns - Element Key (csv: Elmnt Key, json: elementkey) and Side of Street (csv: Side, json: sideofstreet). Element key will be the main column to lookup values across both datasets.

In [ ]:
# Check unique values in all columns in csv first
for col_name in csv_df:
    print(f"Unique values in column: {col_name}")
    csv_df.select(col_name).distinct().show(50, truncate=False)
    print(" ")

In [ ]:
# Check unique values in all columns in json first
for col_name in json_df:
    print(f"Unique values in column: {col_name}")
    json_df.select(col_name).distinct().show(50, truncate=False)
    print(" ")

As seen in the unique values of string columns extracted from the csv file, we observe that the data quality of the datasets is not the best.

In the **csv** file:
- `Study_Area`: Contains many inconsistent and likely manually entered variations of related areas. Standardisation is required for reliable area-level analysis.
- `Side`: Expected direction codes (e.g. N, S, E, W etc.) are mixed with other numbers and letters. Since the json file contains a cleaner `sideofstreet` column, we will clean this column with a lookup to the json data.
- `Total_Vehicle_Count`: Contains invalid negative values - these rows will be removed.
- `Construction`, `Event Closure`, `Peak Hour`: These are supposed to be boolean columns, and also include inconsistent formats (e.g. "Yes" vs "yes"). Standardisation is required here.

In the **json** file:
- `durationinminutes`: There are negative values here which do not make sense - these rows will be removed.

### **2.1. Fixing Specific Columns**

To minimise the changes made to the original dataset, we first create new dataframes to store the cleaned data.
- `cleaned_csv_df`
- `cleaned_json_df`

In [ ]:
cleaned_csv_df = csv_df
cleaned_json_df = json_df

#### **2.1.1. csv**

#### *A. Clean `Study_Area` column (csv)*

In [ ]:
csv_df.select("Study_Area").distinct()\
                            .orderBy("Study_Area")\
                            .show(csv_df.select("Study_Area").distinct().count(), truncate=False)

In [ ]:
# Create a new column - Study_Area_cleaned_L2
cleaned_csv_df = csv_df.withColumn("Study_Area_cleaned_L2", F.col("Study_Area"))

In [ ]:
# Remove invalid values
invalid_values = [" 1-6p; x2\"",
                  " 3-6p",
                  " carpool 7-10 am\"",
                  " if anyone parked there between 7-5:30 I marked it as illegal.\"",
                  "No",
                  "Yes",
                  None]

cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.when(F.col("Study_Area_cleaned_L2").isin(invalid_values), None)
                                           .otherwise(F.col("Study_Area_cleaned_L2")))

cleaned_csv_df.select("Study_Area_cleaned_L2")\
                .distinct()\
                .orderBy("Study_Area_cleaned_L2")\
                .show(cleaned_csv_df.select("Study_Area_cleaned_L2").distinct().count(), truncate=False)

In [ ]:
# Standardise all dashes first
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.regexp_replace("Study_Area_cleaned_L2", "–", "-"))

# Remove everything after dash
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.split("Study_Area_cleaned_L2", "-").getItem(0))

# Remove anything with parentheses: "(text)"
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.split("Study_Area_cleaned_L2", "\\(").getItem(0))

# Remove the number 2017 after area name
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.regexp_replace("Study_Area_cleaned_L2", "2017", ""))

# Remove any extra spaces
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.trim(F.col("Study_Area_cleaned_L2")))

# Standardise all to lowercase
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                                           F.lower(F.col("Study_Area_cleaned_L2")))

cleaned_csv_df.select("Study_Area_cleaned_L2")\
                .distinct()\
                .orderBy("Study_Area_cleaned_L2")\
                .show(cleaned_csv_df.select("Study_Area_cleaned_L2").distinct().count(), truncate=False)

In [ ]:
# Further adjust certain values
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L2",
                    F.when(F.col("Study_Area_cleaned_L2") == "12th ave", "12th avenue")
                    .when(F.col("Study_Area_cleaned_L2") == "12th ave  annual study", "12th avenue")
                    .when(F.col("Study_Area_cleaned_L2") == "chinatown/id", "chinatown id")
                    .when(F.col("Study_Area_cleaned_L2") == "greenlake", "green lake")
                    .otherwise(F.col("Study_Area_cleaned_L2")))

cleaned_csv_df.select("Study_Area_cleaned_L2")\
                .distinct()\
                .orderBy("Study_Area_cleaned_L2")\
                .show(cleaned_csv_df.select("Study_Area_cleaned_L2").distinct().count(), truncate=False)

In [ ]:
# Create a Study_Area_cleaned_L1 to capture groups of related names
cleaned_csv_df = cleaned_csv_df.withColumn("Study_Area_cleaned_L1",
                    F.when(F.col("Study_Area_cleaned_L2").startswith("ballard"), "ballard")
                    .when(F.col("Study_Area_cleaned_L2").startswith("commercial core"), "commercial core")
                    .when(F.col("Study_Area_cleaned_L2").startswith("denny triangle"), "denny triangle")
                    .when(F.col("Study_Area_cleaned_L2").startswith("uptown"), "uptown")
                    .otherwise(F.col("Study_Area_cleaned_L2")))

cleaned_csv_df.select("Study_Area_cleaned_L1")\
                .distinct()\
                .orderBy("Study_Area_cleaned_L1")\
                .show(cleaned_csv_df.select("Study_Area_cleaned_L1").distinct().count(), truncate=False)

In [ ]:
cleaned_csv_df.select("Study_Area_cleaned_L2", "Study_Area_cleaned_L1").toPandas()

#### *B. Clean `Side` column (csv)*

Since the `sideofstreet` column in the json dataset is clean (as observed from the unique values extracted in Part 2 above), we will clean the `Side` column in the csv dataset by looking up to the data in the json dataset.

In [ ]:
# Create a lookup table from json data first
json_lookup = json_df.select("elementkey", "sideofstreet").dropDuplicates(["elementkey"])

# Join csv with json_lookup
cleaned_csv_df = cleaned_csv_df.join(json_lookup,
                                     cleaned_csv_df.Elmntkey == json_lookup.elementkey,
                                     "left")

# Create a new Side_cleaned column
cleaned_csv_df = cleaned_csv_df.withColumn("Side_cleaned", F.col("sideofstreet"))

# Remove lookup columns after creating Side_cleaned column
cleaned_csv_df = cleaned_csv_df.drop("elementkey", "sideofstreet")

# Check final csv_df to make sure Side_cleaned column is created
cleaned_csv_df.limit(5).toPandas()

In [ ]:
# Check unique values of Side_cleaned column
cleaned_csv_df.select("Side_cleaned").distinct().show(50, truncate=False)

In [ ]:
# Check number of NULL values in Side_cleaned column
null_count = cleaned_csv_df.filter(F.col("Side_cleaned").isNull()).count()
null_pct = null_count / cleaned_csv_df.count() * 100

print(null_count)
print(round(null_pct,2), "%")

In [ ]:
# Compare Side and Side_cleaned columns
cleaned_csv_df.select("Elmntkey", "Side", "Side_cleaned").show(50, truncate=False)

Notice that there are certain rows with NULL values in `Side_cleaned` column but with valid values in `Side` column - This is likely because these element keys in csv dataset is not present in the json dataset. We will fill these NULL values with their corresponding values in the `Side` column, should their values be valid. Otherwise, they remain as NULL.

In [ ]:
# Fill NULL values with valid Side values where applicable
Side_valid = ["N", "S", "E", "W", "NE", "NW", "SE", "SW"]

cleaned_csv_df = cleaned_csv_df.withColumn("Side_cleaned",
                                           F.when( F.col("Side_cleaned").isNull() & F.col("Side").isin(Side_valid),
                                                  F.col("Side"))
                                           .otherwise(F.col("Side_cleaned")))

# Compare Side and Side_cleaned columns again
cleaned_csv_df.select("Elmntkey", "Side", "Side_cleaned").show(50, truncate=False)

In [ ]:
# Check NULL values in Side_cleaned column
null_count = cleaned_csv_df.filter(F.col("Side_cleaned").isNull()).count()
null_pct = null_count / cleaned_csv_df.count() * 100

print(null_count)
print(round(null_pct,2), "%")

We have managed to reduce the proportion of NULL values to 1.33%. As such, we can remove these rows to ensure overall data cleanliness.

In [ ]:
# Remove NULL values
cleaned_csv_df = cleaned_csv_df.filter(F.col("Side_cleaned").isNotNull())

# Check that there are no NULL values in Side_cleaned column
cleaned_csv_df.filter(F.col("Side_cleaned").isNull()).count()

In [ ]:
cleaned_csv_df.limit(5).toPandas()

#### *C. Clean `Total_Vehicle_Count` column (csv)*

In [ ]:
# Convert Total_Vehicle_Count data type to integer first
cleaned_csv_df = cleaned_csv_df.withColumn("Total_Vehicle_Count", F.col("Total_Vehicle_Count").cast("int"))

In [ ]:
# Check the number of negative values
neg_count = cleaned_csv_df.filter(F.col("Total_Vehicle_Count") < 0).count()
neg_pct = neg_count / cleaned_csv_df.count() * 100

print(neg_count)
print(round(neg_pct,3), "%")

Since the negative values only make up 0.003% of the entire dataset, we will remove these rows for data cleanliness.

In [ ]:
# Remove negative values
cleaned_csv_df = cleaned_csv_df.filter(F.col("Total_Vehicle_Count") >= 0)

# Check that there are no negative values in Total_Vehicle_Count column in cleaned_csv_df
cleaned_csv_df.filter(F.col("Total_Vehicle_Count") < 0).count()

In [ ]:
cleaned_csv_df.limit(5).toPandas()

#### *D. Clean `Construction` column (csv)*

In [ ]:
cleaned_csv_df.select("Construction").distinct().show(50)

In [ ]:
# Standardise all the lowercases "yes", "no" and "NULL"
cleaned_csv_df = cleaned_csv_df.withColumn("Construction_cleaned",
                                           F.when(F.lower(F.trim(F.col("Construction"))) == "yes", "yes")
                                           .when(F.lower(F.trim(F.col("Construction"))) == "no",  "no")
                                           .otherwise(None))

# Check values in Construction_cleaned column
cleaned_csv_df.select("Construction", "Construction_cleaned").distinct().show(50, truncate=False)

In [ ]:
# Check unique values in Construction_cleaned column
cleaned_csv_df.select("Construction_cleaned").distinct().show(50)

In [ ]:
# Check NULL values in Construction_cleaned column
null_count = cleaned_csv_df.filter(F.col("Construction_cleaned").isNull()).count()
null_pct = null_count / csv_df.count() * 100

print(null_count)
print(round(null_pct,2), "%")

In [ ]:
# Convert data type to Boolean
cleaned_csv_df = cleaned_csv_df.withColumn("Construction_cleaned",
                                           F.when(F.col("Construction_cleaned") == "yes", True)
                                           .when(F.col("Construction_cleaned") == "no", False)
                                           .otherwise(None))

cleaned_csv_df.limit(5).toPandas()

#### *E. Clean `Event Closure` column (csv)*

In [ ]:
cleaned_csv_df.select("Event Closure").distinct().show(50)

In [ ]:
# Standardise all the lowercases "yes", "no" and "NULL"
cleaned_csv_df = cleaned_csv_df.withColumn("EventClosure_cleaned",
                                           F.when(F.lower(F.trim(F.col("Event Closure"))) == "yes", "yes")
                                           .when(F.lower(F.trim(F.col("Event Closure"))) == "no",  "no")
                                           .otherwise(None))

# Check values in EventClosure_cleaned column
cleaned_csv_df.select("Event Closure", "EventClosure_cleaned").distinct().show(50, truncate=False)

In [ ]:
# Check unique values in EventClosure_cleaned column
cleaned_csv_df.select("EventClosure_cleaned").distinct().show(50)

In [ ]:
# Check NULL values in Construction_cleaned column
null_count = cleaned_csv_df.filter(F.col("EventClosure_cleaned").isNull()).count()
null_pct = null_count / csv_df.count() * 100

print(null_count)
print(round(null_pct,2), "%")

In [ ]:
# Convert data type to Boolean
cleaned_csv_df = cleaned_csv_df.withColumn("EventClosure_cleaned",
                                           F.when(F.col("Construction_cleaned") == "yes", True)
                                           .when(F.col("Construction_cleaned") == "no", False)
                                           .otherwise(None))

In [ ]:
cleaned_csv_df.limit(5).toPandas()

#### *F. Clean `Peak Hour? (Yes or No)` column (csv)*

In [ ]:
cleaned_csv_df.select("Peak Hour? (Yes or No)").distinct().show(50)

In [ ]:
# Standardise all the lowercases "yes", "no" and "NULL"
cleaned_csv_df = cleaned_csv_df.withColumn("PeakHour_cleaned",
                                           F.when(F.lower(F.trim(F.col("Peak Hour? (Yes or No)"))) == "yes", "yes")
                                           .when(F.lower(F.trim(F.col("Peak Hour? (Yes or No)"))) == "no",  "no")
                                           .otherwise(None))

# Check values in PeakHour_cleaned column
cleaned_csv_df.select("Peak Hour? (Yes or No)", "PeakHour_cleaned").distinct().show(50, truncate=False)

In [ ]:
# Check unique values in PeakHour_cleaned column
cleaned_csv_df.select("PeakHour_cleaned").distinct().show(50)

In [ ]:
# Check NULL values in PeakHour_cleaned column
null_count = cleaned_csv_df.filter(F.col("PeakHour_cleaned").isNull()).count()
null_pct = null_count / csv_df.count() * 100

print(null_count)
print(round(null_pct,2), "%")

In [ ]:
# Convert data type to Boolean
cleaned_csv_df = cleaned_csv_df.withColumn("PeakHour_cleaned",
                                           F.when(F.col("PeakHour_cleaned") == "yes", True)
                                           .when(F.col("PeakHour_cleaned") == "no", False)
                                           .otherwise(None))

In [ ]:
cleaned_csv_df.limit(5).toPandas()

#### *G. Clean `Date Time` (csv)*

In [ ]:
# Convert data type to Time Stamp first
cleaned_csv_df = cleaned_csv_df.withColumn("DateTime_cleaned", F.to_timestamp("Date Time", "M/d/yyyy H:mm"))

# Create new column for Date
cleaned_csv_df = cleaned_csv_df.withColumn("Date", F.to_date("DateTime_cleaned"))

# Create new column for Time
cleaned_csv_df = cleaned_csv_df.withColumn("Time", F.date_format("DateTime_cleaned", "HH:mm"))

# Create new column for Hour
cleaned_csv_df = cleaned_csv_df.withColumn("Hour", F.hour("DateTime_cleaned"))

# Create new column for Year
cleaned_csv_df = cleaned_csv_df.withColumn("Year", F.year("DateTime_cleaned"))

In [ ]:
cleaned_csv_df.limit(5).toPandas()

#### **2.1.2 json - Paid Parking Transaction Data**

#### *A. Clean `durationinminutes` (json)*

In [ ]:
# Convert durationinminutes data type to integer first
cleaned_json_df = cleaned_json_df.withColumn("durationinminutes", F.col("durationinminutes").cast("int"))

# Check the number of negative values
neg_count = cleaned_json_df.filter(F.col("durationinminutes") < 0).count()
neg_pct = neg_count / cleaned_json_df.count() * 100

print(neg_count)
print(round(neg_pct,3), "%")

In [ ]:
# Remove negative values
cleaned_json_df = cleaned_json_df.filter(F.col("durationinminutes") >= 0)

# Check that there are no negative values in durationinminutes column in cleaned_json_df
cleaned_json_df.filter(F.col("durationinminutes") < 0).count()

#### *B. Clean `transactiondatetime` (json)*

In [ ]:
# Create new column for Date
cleaned_json_df = cleaned_json_df.withColumn("Date", F.to_date("transactiondatetime"))

# Create new column for Hour
cleaned_json_df = cleaned_json_df.withColumn("Hour", F.hour("transactiondatetime"))

# Create new column for Year
cleaned_json_df = cleaned_json_df.withColumn("Year", F.year("transactiondatetime"))

### **2.2 Drop columns that are no longer needed**

In [ ]:
cleaned_csv_df.limit(5).toPandas()

In [ ]:
columns_to_drop = ["Study_Area",
                   "Date Time",
                   "Side",
                   "Construction",
                   "Event Closure",
                   "Study Year",
                   "Peak Hour? (Yes or No)"]

cleaned_csv_df = cleaned_csv_df.drop(*columns_to_drop)

In [ ]:
cleaned_csv_df.printSchema()
cleaned_json_df.printSchema()

### **2.3 Convert remaining columns to appropriate data types**

In [ ]:
# Fix Data Types for cleaned_csv_df
cleaned_csv_df = cleaned_csv_df \
                    .withColumn("Elmntkey", F.col("Elmntkey").cast("int"))\
                    .withColumn("Parking_Spaces", F.col("Parking_Spaces").cast("int"))\
                    .withColumn("Dp_Count", F.col("Dp_Count").cast("int"))

# Check data type after converting
cleaned_csv_df.printSchema()

In [ ]:
# Fix Data Types for cleaned_json_df
cleaned_json_df = cleaned_json_df \
                    .withColumn("amount_paid",F.translate("amount_paid", "$", "").cast("double"))\
                    .withColumn("elementkey", F.col("elementkey").cast("int"))\
                    .withColumn("latitude", F.col("latitude").cast("double"))\
                    .withColumn("longitude", F.col("longitude").cast("double"))\
                    .withColumn("meter_code", F.col("meter_code").cast("int"))\
                    .withColumn("parkingspacenumber", F.col("parkingspacenumber").cast("int"))\
                    .withColumn("transaction_id", F.col("transaction_id").cast("int"))\
                    .withColumn("transactiondatetime", F.to_timestamp("transactiondatetime"))

# Check data type after converting
cleaned_json_df.printSchema()

### **2.4 Check for duplicate rows in both datasets**

In [ ]:
cleaned_csv_df = cleaned_csv_df.dropDuplicates()
cleaned_csv_df.count()

In [ ]:
cleaned_json_df = cleaned_json_df.dropDuplicates()
cleaned_json_df.count()

### **2.5 Check for missing rows in both dataset**

In [ ]:
# Check for number of missing values in each column in csv
from pyspark.sql.functions import col, sum as spark_sum
missing_values = []

for col_name in cleaned_csv_df.columns:
    missing_values.append(spark_sum(col(col_name).isNull().cast("int")).alias(col_name))

missing_values_count = cleaned_csv_df.select(missing_values)
missing_values_count.toPandas()

In [ ]:
# Examine missing values in Elmntkey
cleaned_csv_df.filter(F.col("Elmntkey").isNull()).toPandas()

In [ ]:
# Drop missing values in Elmntkey
cleaned_csv_df = cleaned_csv_df.dropna(subset=["Elmntkey"])

In [ ]:
# Examine missing values in Parking_Spaces
cleaned_csv_df.filter(F.col("Parking_Spaces").isNull()).toPandas()

In [ ]:
# Drop missing values in Parking_Spaces
cleaned_csv_df = cleaned_csv_df.dropna(subset=["Parking_Spaces"])
cleaned_csv_df.count()

In [ ]:
# Drop any rows that are entirely empty in csv
cleaned_csv_df = cleaned_csv_df.na.drop("all")
cleaned_csv_df.count()

In [ ]:
# Check for number of missing values in each column in json
from pyspark.sql.functions import col, sum as spark_sum
missing_values = []

for col_name in cleaned_json_df.columns:
    missing_values.append(spark_sum(col(col_name).isNull().cast("int")).alias(col_name))

missing_values_count = cleaned_json_df.select(missing_values)
missing_values_count.toPandas()

In [ ]:
# Drop any rows that are entirely empty
cleaned_json_df = cleaned_json_df.na.drop("all")
cleaned_json_df.count()

## **3. Data Analysis:** Overview of our Datasets and Variables

Before addressing the business questions, it is crucial that we explore and understand the datasets we are working with.

### **3.1 Numerical Variables**

##### **3.1.1 csv - Annual Parking Study Data**

In [ ]:
# Summary Statistics for numerical columns
from pyspark.sql.types import NumericType

csv_numeric_cols = []

for field in cleaned_csv_df.schema.fields:
    if isinstance(field.dataType, NumericType):
        csv_numeric_cols.append(field.name)

cleaned_csv_df.select(csv_numeric_cols).describe().toPandas()

We will analyse further the relationship between the number of available parking spaces (`Parking_Spaces`) and the number of parked vehicles (`Total_Vehicle_Count`).

In [ ]:
# Correlation between Parking_Spaces and Total_Vehicle_Count
print(cleaned_csv_df.corr("Parking_Spaces", "Total_Vehicle_Count"))

# Scatterplot of Parking_Spaces vs Total_Vehicle_Count
sns.scatterplot(data=cleaned_csv_df.select("Parking_Spaces", "Total_Vehicle_Count").toPandas(), 
                x="Parking_Spaces",
                y="Total_Vehicle_Count")

There is a strong correlation between the number of parking spaces available and the number of parked vehicles, which is aligned with expectation. Logically, we will now proceed to understand the **occupancy rate, which reflects the utilisation**.

In [ ]:
# Create new column and calculate the Occupancy Rate
cleaned_csv_df = cleaned_csv_df.withColumn("Occupancy_Rate",
                                           cleaned_csv_df["Total_Vehicle_Count"] / cleaned_csv_df["Parking_Spaces"])

In [ ]:
# Correlation between Parking_Spaces and Occupancy_Rate
print(cleaned_csv_df.corr("Parking_Spaces", "Occupancy_Rate"))

# Scatterplot of Parking_Spaces vs Occupancy_Rate
sns.scatterplot(data=cleaned_csv_df.select("Parking_Spaces", "Occupancy_Rate").toPandas(), 
                x="Parking_Spaces",
                y="Occupancy_Rate")

The weak correlation between parking spaces and occupancy rate confirms that occupancy is appropriately normalised and utilisation is not simply driven by size of parking spaces. This indicates that the different utilisation rates reflect the true variation in parking demand - we will therefore explore further how demand pressures varies across Seattle and identify possible strategies to optimise parking spaces in Seattle. 

In [ ]:
# Summary Statistics for selected columns
cleaned_csv_df.select("Parking_Spaces", "Total_Vehicle_Count", "Occupancy_Rate").describe().toPandas()

In [ ]:
cleaned_csv_df.groupBy("Parking_Spaces") \
                .agg(F.avg("Total_Vehicle_Count").alias("avg_vehicle_count"),
                    F.avg("Occupancy_rate").alias("avg_occupancy_rate")) \
                .orderBy("Parking_Spaces") \
                .show(cleaned_csv_df.select("Parking_Spaces").distinct().count())

##### **3.1.2 json - Paid Parking Transaction Data**

In [ ]:
# Summary Statistics for numerical columns
json_numeric_cols = []

for field in cleaned_json_df.schema.fields:
    if isinstance(field.dataType, NumericType):
        json_numeric_cols.append(field.name)

cleaned_json_df.select(json_numeric_cols).describe().toPandas()

In [ ]:
# Correlation between amount_paid and durationinminutes
cleaned_json_df.corr("amount_paid", "durationinminutes")

In [ ]:
# Scatterplot of amount_paid vs durationinminutes
sns.scatterplot(data=cleaned_json_df.select("amount_paid", "durationinminutes").toPandas(), 
                x="durationinminutes",
                y="amount_paid")

The correlation between `amount_paid` and `durationinminutes` is moderately weak - While longer duration does correspond with slightly higher amounts paid, the relationship is noisy. This is likely because this dataset captures payment transactions, and hence the payment amount might not capture the full amount paid for the parking stay. For example, a single parking trip might involve multiple types of payments (e.g. via meter and then via phone).

Since the dataset does not include a unique identifier linking each transaction to a specific driver or parking session, the recorded amount paid does not necessarily represent the total amount paid for the associated stay. Additionally, pricing structures for parking spaces often adopt economies of scale in reality, which further weakens the linear relationship between duration and paymnet amount.

Given these limitations, we will not focus on the price vs duration as this could lead to misleading results. 

## *3.2 Categorical Variables*

##### ***3.2.1 csv - Annual Parking Study Data***

We will be analysing the following categorical variables:
- `Study_Area_cleaned_L2` and `Study_Area_cleaned_L1`
- `Side_cleaned`
- `Construction_cleaned`
- `EventClosure_cleaned`
- `PeakHour_cleaned`
- `Date`

In [ ]:
# Count by Study_Area_cleaned_L2
cleaned_csv_df.groupBy("Study_Area_cleaned_L2").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Count by Study_Area_cleaned_L1
cleaned_csv_df.groupBy("Study_Area_cleaned_L1").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Study Area by Mapping Levels
cleaned_csv_df.select("Study_Area_cleaned_L1","Study_Area_cleaned_L2").distinct().orderBy("Study_Area_cleaned_L1").toPandas()

In [ ]:
# Count by Side_cleaned
cleaned_csv_df.groupBy("Side_cleaned").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Count by Construction_cleaned
cleaned_csv_df.groupBy("Construction_cleaned").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Count by EventClosure_cleaned
cleaned_csv_df.groupBy("EventClosure_cleaned").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Count by PeakHour_cleaned
cleaned_csv_df.groupBy("PeakHour_cleaned").count().orderBy("count", ascending=False).toPandas()

In [ ]:
# Count by Hour
cleaned_csv_df.groupBy("Hour").count().orderBy("Hour").toPandas()

From the 2 outputs above, we can conclude that the data in `PeakHour_cleaned` is not reliable for further analysis. Typically, a peak hour period falls between 07:00 - 09:00 / 17:00 - 19:00, however we are seeing a mismatch in results between the Count by PeakHour_cleaned and Count by Hour. This mismatch could be due to human error or wrong/ subjective understanding of peak hour by the data collectors. To answer the business questions below, we will therefore not analyse by the PeakHour_cleaned parameter. 

In [ ]:
# Count by Date
cleaned_csv_df.groupBy("Date").count().orderBy("count", ascending=False).toPandas()

The analysis of the data above does not yield any immediate insights. We will create separate columns for month and day of the week to enable more meaningful analysis. 

In [ ]:
# Create Month and Month_No columns
cleaned_csv_df = cleaned_csv_df.withColumn("Month", F.date_format("Date", "MMM"))
cleaned_csv_df = cleaned_csv_df.withColumn("Month_No", F.month("Date"))

cleaned_csv_df.limit(5).toPandas()

In [ ]:
# Create DayOfWeek and DayOfWeek_No columns
cleaned_csv_df = cleaned_csv_df.withColumn("DayOfWeek", F.date_format("Date", "E"))
cleaned_csv_df = cleaned_csv_df.withColumn("DayofWeek_No", F.dayofweek("Date"))

cleaned_csv_df.limit(5).toPandas()

As shown in the above extract, the dayofweek() function follows the US convention, the day numbering begins on Sunday - Sunday = 1, Monday = 2 ... Saturday = 7. 

In [ ]:
cleaned_csv_df.select("Month").distinct().show()

In [ ]:
# Count by Month
cleaned_csv_df.groupBy("Month").count().orderBy("count", ascending=False).limit(12).toPandas()

The extract above shows that the data is skewed towards the months in Spring, consistent with the information on the website. Hence, we will not proceed with any analysis based on the month parameter, as the dataset contains inherent bias.

In [ ]:
cleaned_csv_df.select("DayOfWeek").distinct().show()

In [ ]:
# Count by DayOfWeek
cleaned_csv_df.groupBy("DayOfWeek").count().orderBy("count", ascending=False).toPandas()

Similar to the month-level analysis, we observe that the dataset heavily has data on mid-week (Tuesday to Thursday), with far fewer observations on weekends. Additionally, there are no observations for Mondays or Fridays. Given this inherent bias, we will not perform analyses based on days of the week or compare weekends with weekdays. 

##### ***3.2.2 json - Paid Parking Transaction Data***

In [ ]:
# Count by sideofstreet
cleaned_json_df.groupBy("sideofstreet").count().orderBy("count",ascending=False).toPandas()

In [ ]:
# Count by Date
cleaned_json_df.groupBy("Date").count().orderBy("count", ascending=False).toPandas()

The json dataset contains only the most recent week of data from the date of extraction, and does not contain any historical data with timeframe that coincides exactly with the csv dataset. As there are no overlaps in timeframes, direct comparisons between the 2 datasets are not possible. However, we can still derive useful insights by examining and cross-referencing fields that exist in both datasets using targeted lookups and validation where appropriate. 

## **4. Answering The Main Business Question:** `How should we optimise parking spaces across Seattle?`

We will address the main business question using a three-step approach, while taking into considerations the characteristics and limitations of the datasets identified in Part 3.

### **4.1. How does utilisation vary across Seattle?**

To address this question, we will examine parking availability and demand across Seattle using several dimensions: by study area, by side of the street, and by considering the impacts of construction and event closures. 

#### *A. Occupancy by Side of Street*

In [ ]:
cleaned_csv_df.groupBy("Side_cleaned") \
    .agg(F.sum("Parking_Spaces").alias("Total_Parking_Spaces"),
         F.avg("Total_Vehicle_Count").alias("Avg_Vehicle_Count"),
         F.avg("Occupancy_Rate").alias("Avg_Occupancy_Rate")) \
    .orderBy(F.col("Avg_Occupancy_Rate").desc()) \
    .toPandas()

**Insights:** The occupancy rate across all sides of the street remains relatively similar - ranging from 0.73 to 0.77. We can conclude that the side of street does not influence demand.

#### *B. Impact of Construction*

In [ ]:
cleaned_csv_df.groupBy("Construction_cleaned") \
                .agg(F.count("*").alias("Count"),
                     F.avg("Total_Vehicle_Count").alias("Avg_Vehicle_Count"),
                     F.avg("Occupancy_Rate").alias("Avg_Occupancy_Rate"))\
                .filter(F.col("Construction_cleaned").isNotNull())\
                .orderBy("Avg_Vehicle_Count", ascending=False).toPandas()

**Insights:** Construction has a significant impact on utilisation, reducing it from 0.77 to 0.31.

#### *C. Impact of Event Closure*

In [ ]:
cleaned_csv_df.groupBy("EventClosure_cleaned") \
                .agg(F.count("*").alias("Count"),
                     F.avg("Total_Vehicle_Count").alias("Avg_Vehicle_Count"),
                     F.avg("Occupancy_Rate").alias("Avg_Occupancy_Rate"))\
                .filter(F.col("EventClosure_cleaned").isNotNull())\
                .orderBy("Avg_Vehicle_Count", ascending=False).toPandas()

**Insights:** Similar to construction, the occurrence of an event also reduces utilisation, lowering it from 0.77 to 0.31.

#### *D. Occupancy by Study Area*

In [ ]:
Occ_StudyArea = cleaned_csv_df.groupBy("Study_Area_cleaned_L1") \
                        .agg(F.avg("Parking_Spaces").alias("Avg_Parking_Spaces"),
                             F.avg("Total_Vehicle_Count").alias("Avg_Vehicle_Count"),
                             F.avg("Occupancy_Rate").alias("Avg_Occupancy_Rate")) \
                        .orderBy(F.col("Avg_Occupancy_Rate").desc())
                        
Occ_StudyArea.toPandas()

**Insights:** 15th Avenue is the only area with an occupancy rate > 1, indicating that more vehicles were parked than the number of available spaces - likely reflects illegal parking activity and suggests that 15th Avenue is oversubscribed.

Areas with high utilisation (occupancy rate > 0.8) include First Hill, Dexter, Cherry Hill and Green Lake. These results align with the characteristics of these neighbourhoods: First Hill and Cherry Hill are medical hubs, Dexter has dense residential and commercial development, and Green Lake is a popular recreational area.

In contrast, Roosevelt, Little Saigon and Lake City show low utilisation (occupancy rate < 0.7), with Little Saigon and Lake city operating at less 50% occupancy. Notably, Little Saigon also has one of the highest numbers of available parking spaces.

***While utilisation is clearly impacted by construction and events, the largest variation in occupancy rates is observed across study areas. Therefore we will focus on study area as the primary dimension to draw actionable insights for optimising parking spaces.***

In [ ]:
# Capture the areas with high occupancy
HighOcc_StudyArea = Occ_StudyArea.filter(F.col("Avg_Occupancy_Rate") > 0.80)
HighOcc_StudyArea.toPandas()

In [ ]:
# Capture the areas with low occupancy
LowOcc_StudyArea = Occ_StudyArea.filter(F.col("Avg_Occupancy_Rate") < 0.70)
LowOcc_StudyArea.toPandas()

In [ ]:
HighOcc_StudyArea_List = []
for i in HighOcc_StudyArea.collect():
    HighOcc_StudyArea_List.append(i["Study_Area_cleaned_L1"])

LowOcc_StudyArea_List = []
for i in LowOcc_StudyArea.collect():
    LowOcc_StudyArea_List.append(i["Study_Area_cleaned_L1"])

#### *E. Occupancy by Study Area and Construction*

To deepen our analysis by study areas, we will examine the impact of construction on the above two groups. In general, construction works have a persistent and structural impact (often lasting months at least) and affect driving routes, parking availability and commuting behaviour. 

In [ ]:
Occ_StudyArea_Construction = cleaned_csv_df.groupBy("Study_Area_cleaned_L1", "Construction_cleaned") \
                        .agg(F.avg("Parking_Spaces").alias("Avg_Parking_Spaces"),
                             F.avg("Total_Vehicle_Count").alias("Avg_Vehicle_Count"),
                             F.avg("Occupancy_Rate").alias("Avg_Occupancy_Rate")) \
                        .orderBy(F.col("Avg_Occupancy_Rate").desc())

In [ ]:
HighOcc_StudyArea_Construction = Occ_StudyArea_Construction\
                                    .filter(F.col("Study_Area_cleaned_L1").isin(HighOcc_StudyArea_List))\
                                    .filter(F.col("Construction_cleaned").isNotNull())\
                                    .orderBy("Study_Area_cleaned_L1")

HighOcc_StudyArea_Construction.toPandas()

In [ ]:
LowOcc_StudyArea_Construction = Occ_StudyArea_Construction\
                                    .filter(F.col("Study_Area_cleaned_L1").isin(LowOcc_StudyArea_List))\
                                    .filter(F.col("Construction_cleaned").isNotNull())\
                                    .orderBy("Study_Area_cleaned_L1")

LowOcc_StudyArea_Construction.toPandas()

**Insights:** The results are broadly consistent with earlier findings that construction suppresses utilisation. In high-utilisation areas such as First Hill, the occupancy rate reaches around 0.90 when construction is absent. This indicates possible operation saturation where targeted increases in supply may be necessary to better accommodate the underlying demand. 

In contrast, low-utilisation areas maintain low occupancy rates even without construction, which reinforces the conclusion that these areas may present opportunities to repurpose the space or consider price reductions.

### **4.2. How has utilisation varied across the years?**

Now that we have established the high-utilisation and low-utilisation study areas based on the 2014 - 2019 aggregated data, we will now examine the yearly utilisation rates.

In [ ]:
Year_StudyArea = (cleaned_csv_df.groupBy("Study_Area_cleaned_L1", "Year")
                    .agg(F.avg("Occupancy_Rate").alias("Avg_Occupancy")))

Year_StudyArea = (Year_StudyArea.withColumn("UtilCategory",
                                            F.when(F.col("Avg_Occupancy") > 0.80, "High")
                                            .when(F.col("Avg_Occupancy") < 0.70, "Low")
                                            .otherwise("Mid")))

In [ ]:
UtilCategory_Summary = Year_StudyArea.groupBy("Study_Area_cleaned_L1")\
                                        .pivot("UtilCategory", ["High", "Mid", "Low"]).count()\
                                        .orderBy("High", ascending=False)

UtilCategory_Summary.toPandas()

The above table shows the number of times each area has fallen in different utilisation category bucket. First Hill has consistently been a high-utilisation area for all 6 years (count = 6.0), with Green Lake following closely behind (count = 4.0). For our earlier determined low-utilisation areas, we can see that Lake City and Little Saigon actually only has data for 1 year.

In [ ]:
UtilCategory_Year = (Year_StudyArea
                       .select("Study_Area_cleaned_L1", "Year", "UtilCategory")
                       .groupBy("Study_Area_cleaned_L1")
                       .pivot("Year")
                       .agg(F.first("UtilCategory"))
                       .orderBy("2019", ascending=False))

def color_demand(val):
    if val == "High":
        return "background-color: #b6f2b6"   
    elif val == "Mid":
        return "background-color: #fff799"   
    elif val == "Low":
        return "background-color: #ff9999"   
    else:
        return ""   

UtilCategory_Year_Style = UtilCategory_Year.toPandas()\
                            .style.applymap(color_demand,
                                            subset=[c for c in UtilCategory_Year.toPandas().columns if c != "Study_Area_cleaned_L1"])
UtilCategory_Year_Style

Based on the tables above, we conclude the following study areas as high-demand and low-demand:
- **High-demand**: First Hill, Green Lake, Cherry Hill, West Lake, Uptown, Fremont
  - First Hill: High utilisation for all 6 years
  - Green Lake: High utilisation for 4 out of 6 years
  - Green Lake and Cherry Hill: High utilisation for 3 out of 6 years + overall utilisation > 0.8
  - West Lake, Uptown and Fremont: Moved from Mid utilisation to High utilisation in later years
- **Low-demand**: Roosevelt, Columbia City, Commercial Core
  - Roosevelt: Low utilisation for 5 out of 6 years
  - Columbia City: Low utilisation for later 2 years; had High utilisation in 2015
  - Commercial Core: Moved from Mid utilisation to Low utilisation in the later 2 years

In [ ]:
HighOcc_StudyArea_Final = ["first hill", "green lake", "cherry hill", "westlake", "uptown", "fremont"]
LowOcc_StudyArea_Final = ["roosevelt", "columbia city", "commercial core"]

In [ ]:
HighOcc_StudyArea_Final_List = [area for area in HighOcc_StudyArea_Final]
LowOcc_StudyArea_Final_List = [area for area in LowOcc_StudyArea_Final]

### **4.3. Are prices aligned with these Demand Study Areas?**

Now that we have established the high-demand and low-demand Study Areas, we will proceed to understand the pricing at these areas based on the latest json dataset.

***Note: Since the datasets have no overlaps in time, we assume that there are no drastic changes in utilisation category for each Study Area from 2019 to 2025.***

In [ ]:
# Create a lookup table from csv data first
csv_lookup = cleaned_csv_df.withColumnRenamed("ElmntKey", "elementkey")\
                            .select("elementkey", "Study_Area_cleaned_L1")\
                            .dropDuplicates(["elementkey"])

In [ ]:
# Extract key columns from json data for price analysis
json_PriceAnalysis = cleaned_json_df.select("elementkey", "amount_paid", "durationinminutes", "payment_mean")

In [ ]:
# Left join both tables - join csv_lookup to json_PriceAnalysis (left)
PriceAnalysis_df = json_PriceAnalysis.join(csv_lookup, on="elementkey", how="left")

PriceAnalysis_df.limit(5).toPandas()

In [ ]:
# Checked whether there are significant unmapped study areas
unmapped_count = PriceAnalysis_df.filter(F.col("Study_Area_cleaned_L1").isNull()).count()
unmapped_pct = unmapped_count/ PriceAnalysis_df.count()

print(unmapped_count)
print(unmapped_pct)

#### *A. Prices and Occupancy by Study Areas*

In [ ]:
# Price by Study area for High-demand areas
Price_HighOcc_StudyArea = PriceAnalysis_df.groupBy("Study_Area_cleaned_L1")\
                            .agg(F.avg("amount_paid").alias("Avg_Price"))\
                            .filter(F.lower(F.col("Study_Area_cleaned_L1")).isin(HighOcc_StudyArea_Final_List))\
                            .orderBy("Avg_Price")

Price_HighOcc_StudyArea.toPandas()

In [ ]:
# Price by Study area for Low-demand areas
Price_LowOcc_StudyArea = PriceAnalysis_df.groupBy("Study_Area_cleaned_L1")\
                            .agg(F.avg("amount_paid").alias("Avg_Price"))\
                            .filter(F.lower(F.col("Study_Area_cleaned_L1")).isin(LowOcc_StudyArea_Final_List))\
                            .orderBy("Avg_Price")

Price_LowOcc_StudyArea.toPandas()

## **5. Conclusion:** Connecting All the Dots

To connect the various parts of our analysis together, we can summarise with the following tables:

In [ ]:
Occ_Year = (Year_StudyArea.select("Study_Area_cleaned_L1", "Year", "Avg_Occupancy")
                            .groupBy("Study_Area_cleaned_L1")
                            .pivot("Year")
                            .agg(F.first("Avg_Occupancy"))
                            .orderBy("Study_Area_cleaned_L1"))

In [ ]:
Final_HighOcc = Price_HighOcc_StudyArea\
        .join(Occ_Year, "Study_Area_cleaned_L1", "left")\
        .join(Occ_StudyArea.select("Study_Area_cleaned_L1",
                                   F.col("Avg_Occupancy_Rate").alias("Avg_Occ (2014 - 2019)")),
              "Study_Area_cleaned_L1",
              "left")\
        .orderBy("2019", ascending=False)

Final_HighOcc = (Final_HighOcc.withColumnRenamed("Study_Area_cleaned_L1", "High_Demand_Area")
                                 .withColumnRenamed("Avg_Price", "Avg_Price_2025")
                                 .withColumnRenamed("2014", "Occ_2014")
                                 .withColumnRenamed("2015", "Occ_2015")
                                 .withColumnRenamed("2016", "Occ_2016")
                                 .withColumnRenamed("2017", "Occ_2017")
                                 .withColumnRenamed("2018", "Occ_2018")
                                 .withColumnRenamed("2019", "Occ_2019"))

Final_HighOcc.toPandas()

In [ ]:
HighOcc_StudyArea_Trend = Year_StudyArea.filter(F.col("Study_Area_cleaned_L1")
                                                 .isin(HighOcc_StudyArea_Final))\
                           .orderBy("Study_Area_cleaned_L1", "Year")

plt.figure(figsize=(10,5))
sns.lineplot(
    data=HighOcc_StudyArea_Trend.toPandas(),
    x="Year",
    y="Avg_Occupancy",
    hue="Study_Area_cleaned_L1",
    marker="o")

plt.title("Occupancy for High-Demand Areas Each Year")
plt.ylabel("Average Occupancy Rate")
plt.xlabel("Year")
plt.legend(title="Study Area")
plt.grid(True)
plt.show()

In [ ]:
LowOcc_StudyArea_Trend = Year_StudyArea.filter(F.col("Study_Area_cleaned_L1")
                                                 .isin(LowOcc_StudyArea_Final))\
                           .orderBy("Study_Area_cleaned_L1", "Year")

plt.figure(figsize=(10,5))
sns.lineplot(
    data=LowOcc_StudyArea_Trend.toPandas(),
    x="Year",
    y="Avg_Occupancy",
    hue="Study_Area_cleaned_L1",
    marker="o")

plt.title("Occupancy for Low-Demand Areas Each Year")
plt.ylabel("Average Occupancy Rate")
plt.xlabel("Year")
plt.legend(title="Study Area")
plt.grid(True)
plt.show()

In [ ]:
Final_LowOcc = Price_LowOcc_StudyArea\
        .join(Occ_Year, "Study_Area_cleaned_L1", "left")\
        .join(Occ_StudyArea.select("Study_Area_cleaned_L1",
                                   F.col("Avg_Occupancy_Rate").alias("Avg_Occ (2014 - 2019)")),
              "Study_Area_cleaned_L1",
              "left")\
        .orderBy("2019", ascending=False)

Final_LowOcc = (Final_LowOcc.withColumnRenamed("Study_Area_cleaned_L1", "Low_Demand_Area")
                                 .withColumnRenamed("Avg_Price", "Avg_Price_2025")
                                 .withColumnRenamed("2014", "Occ_2014")
                                 .withColumnRenamed("2015", "Occ_2015")
                                 .withColumnRenamed("2016", "Occ_2016")
                                 .withColumnRenamed("2017", "Occ_2017")
                                 .withColumnRenamed("2018", "Occ_2018")
                                 .withColumnRenamed("2019", "Occ_2019"))

Final_LowOcc.toPandas()

### **5.1. Recommendations**

Combining our analyses of historical data (2014 to 2019) and recent transaction data in 2025, we segment into 4 actionable categories:

**Category 1: Demand is High and Prices are High:** *First Hill, Fremont*

Prices for these areas are consistent with the strong demand, signalling an appropriate pricing.

**Category 2: Demand is High and Prices are Lower:** *Cherry Hill, Green Lake, Uptown and Westlake*

These high-demand areas have modest prices and lower prices than that of Category 1. This signals opportunities to revise prices upwards which could potentially improve parking turnover. Since utilisation remains high, a slight increase in prices here would likely increase overall revenue.

**Category 3: Demand is Low and Prices are High:** *Commercial Core, Columbia City*

These areas do not show persistent demand and yet prices are high - It is likely that the higher prices have dampened an already lacklustre demand in these areas. This signals 2 opportunities:
   A. Revise prices downwards: To encourage organic demand and improve occupancy rates
   B. Reduce parking spaces and repurpose them for other public uses

**Category 4: Demand is Low and Prices are Low:** *Roosevelt*
The lower prices match an already low demand in the area.

### **5.2. Limitations**

Both datasets present several limitations. Key issues include:
- **Different timeframes**: The csv dataset covers the period 2014 to 2019, while the json dataset only spans a week in 2025 (29 November 2025 to 5 December 2025).
- **Sampling bias**: The csv dataset is mostly from Tuesdays to Thursdays and in the Spring and Summer months only.
- **Data input quality**: Inconsistency in data inputs, likely due to manual data entry. 

### **5.3. Next Steps: Possible Further Analyses**

Once more comprehensive and complete datasets are made available, possible further analyses include:
- Expanding on time-dimension analysis, such as analysing how utilisation differ across months of the year, days of the week or over the past decade etc.
- Expanding on price-dimension analysis, such as analysing total parking fees per visit and deriving more insights on the price elasticity
- Expanding on other factors that could affect parking spaces such as the proximity to supermarkets, tourist attractions etc.

### **5.4. Overall Conclusion**

Overall, aligning parking rates with structural demand patterns provides a feasible path to help Seattle achieve more optimised parking spaces. High-demand areas could benefit from offering higher prices, while low-demand areas could either lower prices to encourage higher utilisation or repurpose these vacant places for other public uses.

We hope you have enjoyed our analysis and detailed walkthrough of our thought process and approach. :)